In [1]:
# ruff: noqa

## 0. Imports

In [1]:
import json
import warnings
import gc
import os
from datetime import datetime, timedelta

import folium as fl
import geopandas as gpd
import h3
import hvplot.pandas
import matplotlib.pyplot as plt
import movingpandas as mpd
import numpy as np
import pandas as pd
import shapely as shp
from bson import json_util
from geopandas import GeoDataFrame, read_file
from holoviews import opts
from movingpandas import TrajectoryCollection
from shapely.geometry import LineString, Point, Polygon, shape

/workspaces/python_samples/.venv/lib/python3.13/site-packages/movingpandas/__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


# Table of Contents
- [0. Imports](#0-imports)
- [1. Introduction](#1-introduction)
- [2. Reading data](#2-reading-data)
  - [2.1 Checking format of raw data](#21-checking-format-of-raw-data)
  - [2.2 Data preprocessing](#22-data-preprocessing)
- [3. Trajectory ID's](#3-trajectory-ID's)
  - [3.1 Defining assign_trajectory_ids()](#31-defining-assign_trajectory_ids-())
  - [3.2 Assigning initial trajectory ID](#32-assigning-initial-trajectory-id)
  - [3.3 Printing results (debugging)](#33-printing-results-(debugging))
  - [3.4 Removing same time events within trajectories](#34-removing-same-time-events-within-trajectories)
- [4. Splitting trajectories on observationgap](#4-splitting-trajectories-on-observationgap)
  - [4.1 Preparing sample data (two week interval)](#4-1-preparing-sample-data-two-week-interval)
  - [4.2 Splitting trajectories](#42-splitting-trajectories)
  - [4.3 Plotting distance loss](#43-plotting-distance-loss)



## 1 Introduction


This notebook documents the process of assigning trajectory IDs and evaluating the effect/dataloss of splitting trajectories at different observation gap thresholds.  

First, we define an algorithm that generates consistent `trajectory_id`s based on event sequences for each device. This ensures that each trajectory is uniquely and consistently identified, making it possible to analyze and compare how trajectories change when applying different splitting thresholds.

Next, we analyze how splitting trajectories at varying observation gaps impacts the total trajectory length. By plotting the percentage loss in distance alongside the number of resulting trajectories, we aim to identify a threshold that balances data continuity with accurate segmentation and limited loss of data.

## 2 Reading data

In this part we check the formatting of the raw data before selecting relevant data and converting to a df

### 2.1 Checking format of raw data

The following functions are defined to check consistency of the variables used for analysis: ['device_id', 'recordedAt', 'odometer', 'location']

has_device_id(record):

Checks whether a record has a key called "deviceId" and if that is a string with length more than 0.


has_recordAt(record):

Chech whether a record has a key called "recordedAt" with a dict containing the key "$date" ensuring all values are in the same format. 


has_odometer(record):

Check whether a record has a key called "odometer" within "details".


is_val_idpoint(record):

checks whether "location" exists and is a dict.

If that contain a key "type" with a value "Point".

then checks if "coordinates" exists, and is a list of length 2.

Both values in "coordinates" are numeric (int or float).





In [ ]:
# Load JSON array from file
with open("synchappgateway.positions.json", "r") as f:
    raw_data = json.load(f)

In [ ]:
# Print key types for first record
first = raw_data[0]
for k, v in first.items():
    print(k, type(v).__name__)

In [ ]:
def has_device_id(record):
    device_id = record.get("deviceId")
    if isinstance(device_id, str) and len(device_id) > 0:
        return True
# Apply to all records

def has_recordedAt(record):
    recorded_at = record.get("recordedAt")
    if not isinstance(recorded_at, dict):
        return False
    if "$date" not in recorded_at:
        return False
    return True

def has_odometer(record):
    details = record.get("details")
    if not isinstance(details, dict):
        return False
    if "odometer" not in details:
        return False
    return True

def is_val_idpoint(record):
    loc = record.get("location")
    if not isinstance(loc, dict):
        return False
    if loc.get("type") != "Point":
        return False
    coords = loc.get("coordinates")
    if not isinstance(coords, list):
        return False
    if len(coords) != 2:
        return False
    # Check that both coordinates are numbers
    if not all(isinstance(c, (int, float)) for c in coords):
        return False
    return True


all_has_device_id = all(has_device_id(r) for r in raw_data)
all_has_recordedAt = all(has_recordedAt(r) for r in raw_data)
all_has_odometer = all(has_odometer(r) for r in raw_data)
all_val_idpoints = all(is_val_idpoint(r) for r in raw_data)

print("All records have valid deviceId:", all_has_device_id)
print("All records have valid recordedAt:", all_has_recordedAt)
print("All records have valid odometer in details:", all_has_odometer)
print("All records have valid GEOJSON Point location:", all_valid_points)


All records have keys with correct formatting. 

NB: Clear kernel before proceeding to load the data. 

### 2.2 Data preprocessing 

In [2]:
# Load JSON array from file
with open("synchappgateway.positions.json", "r") as f:
    data = json.load(f, object_hook=json_util.object_hook)

# Convert to DataFrame
#df = pd.DataFrame(data)
df = pd.DataFrame(data, columns=["_id","location","latitude","longitude","recordedAt", "deviceId", "event", "odometer"])
print("READING DATA TO DF DONE")


#del data
#gc.collect()

df1 = df[(df["latitude"] > 0) & (df["longitude"] > 0)].copy()

#del df
#gc.collect()

#Creating geometry column and setting index
df1["geometry"] = df1["location"].apply(shape)
cols_to_keep = ["_id","recordedAt",
   "deviceId",
   "event",
   "odometer",
   "geometry",
]
df1 = df1[cols_to_keep]
df1.reset_index(drop=True, inplace=True)
print("RELEVANT COLUMNS SELECTED")
print("READING DATA DONE")

READING DATA TO DF DONE
RELEVANT COLUMNS SELECTED
READING DATA DONE


## 3 Defining Trajectories

### 3.1 Defining assign_trajectory_ids()

We define trajectories as follows:
1) A new trajectory will allways be made if reg_started event occurs.

2) A trajectory will always be terminated if reg_complete or reg_postpone occurs.

3) Segments after a reg_complete or reg_postpone wll be their own segments as well.

This is the baseline.

Afterwards we will be splitting based on time between points.

In [3]:
def assign_trajectory_ids_2(data):
    current_device = None
    traj_ids = []
    counter = 0
    device_switch = True
    previous_completed = False
    current_started = False
    for device_id, group in data.groupby("deviceId", group_keys = False):
            print(device_id)
            for _, row in group.iterrows():
                device_switch = current_device != row['deviceId']
                current_started = row['event'] == "reg_started"
                if device_switch or previous_completed or current_started:
                    counter += 1
                    current_device = row["deviceId"]
                previous_completed = row["event"] in ["reg_complete", "reg_postpone"]
                traj_ids.append(f"{counter:06d}")
    return pd.Series(traj_ids, index=data.index)

### 3.2 Assigning initial trajectory ID

In [4]:
# Assigning trajectory ID's
df1 = df1.sort_values(["deviceId", "recordedAt"], ascending=[True, True])
df1["odometer"] = df1["odometer"].replace(0, np.nan)
print('ASSIGNING TRAJECTORY IDS')
df1["trajectory_id"] = assign_trajectory_ids_2(df1)
print("INITIAL TRAJECTORY_IDS ASSIGNED")

# Creating geodataframe using 4326
gdf = GeoDataFrame(df1, geometry="geometry", crs=4326)
print("GEO DATAFRAME DONE")

ASSIGNING TRAJECTORY IDS
ALEWIL-11
ANDEKM-11
BJÖJOH-11
CARÖBE-11 
ELLSTU-11
ERIROH-11
JENLIF-11
JOHSTU-11
KENBEN-11
KENKÄP-11
MAGJÖN-11
MIKLEU-11
MORNIL-11
NIKNIL-11
NIMARE-11
ROBFÄR-11
ROBHEN-11
TEDPET-11
kenkäp-11
INITIAL TRAJECTORY_IDS ASSIGNED
GEO DATAFRAME DONE


### 3.3 Printing results (debugging)

A check is done to determine if the algorithm performs as intended

In [5]:
### DEBUGGIN trajectory_id algorithm with print ###
groups = gdf.groupby("trajectory_id")
#groups = gdf_2weeks.groupby("trajectory_id")
prev_id = []
for value, group in groups:
    device_id = group["deviceId"].iloc[0]
    device_id_ = group["deviceId"].iloc[-1]
    first_event = group["event"].iloc[0]
    first_event_time = group["recordedAt"].iloc[0].floor("s")
    last_event_time = group["recordedAt"].iloc[-1].floor("s")
    last_event = group["event"].iloc[-1]
    dif = int(value) - int(prev_id[-1]) if prev_id else 1
    print(f"|{first_event_time}| {last_event_time}| {device_id}| {device_id_}| ID increase: {dif}| {value}: {first_event} to {last_event}| ")
    prev_id.append(value)

|2025-01-10 17:16:33| 2025-01-20 16:20:41| ALEWIL-11| ALEWIL-11| ID increase: 1| 000001: position to position| 
|2025-01-20 16:21:36| 2025-01-20 17:16:17| ALEWIL-11| ALEWIL-11| ID increase: 1| 000002: reg_started to reg_postpone| 
|2025-01-20 17:16:34| 2025-01-20 17:16:44| ALEWIL-11| ALEWIL-11| ID increase: 1| 000003: position to position| 
|2025-01-20 17:18:35| 2025-01-20 17:21:20| ALEWIL-11| ALEWIL-11| ID increase: 1| 000004: reg_started to reg_postpone| 
|2025-01-20 17:21:30| 2025-01-20 17:21:30| ALEWIL-11| ALEWIL-11| ID increase: 1| 000005: reg_started to reg_postpone| 
|2025-01-20 17:23:00| 2025-01-20 17:23:00| ALEWIL-11| ALEWIL-11| ID increase: 1| 000006: position to position| 
|2025-01-20 17:25:18| 2025-01-20 17:33:22| ALEWIL-11| ALEWIL-11| ID increase: 1| 000007: reg_started to reg_postpone| 
|2025-01-20 17:41:26| 2025-01-20 17:46:59| ALEWIL-11| ALEWIL-11| ID increase: 1| 000008: reg_started to reg_postpone| 
|2025-01-20 17:47:23| 2025-01-20 17:50:09| ALEWIL-11| ALEWIL-11| ID i

### 3.4 Removing same time events within trajectories

3.4.1 First we document that movingpandas drops points at identical timestamps within trajectories. 

3.4.2 Then we proceed with the analysis of duplicates and how we drop them.

We show that all duplicate groups share identical geographical information except in one case (illustraded below (floats?))



 #### 3.4.1 Documenting that MovingPandas drops points at identical timestamp (not part of mpd documentation)

In [6]:
#Creating sample DF
n = 5
coords = (12.34, 56.78)
timestamp = datetime(2025, 9, 2, 12, 0, 0)
constant_id = 1

# Create timestamps: all the same except last one +5 seconds
timestamps = [timestamp] * (n - 1) + [timestamp + timedelta(seconds=5)]

# Create DataFrame
sample_df = pd.DataFrame({
    'row_id': range(1, n + 1),       # Unique row IDs
    'id': [constant_id] * n,         # Constant ID
    'geometry': [Point(coords)] * n, # Same coordinates
    'timestamp': timestamps           # Timestamps
})

# Convert to GeoDataFrame
sample_gdf = gpd.GeoDataFrame(sample_df, geometry='geometry', crs = 4326)


traj = mpd.TrajectoryCollection(sample_gdf, traj_id_col='id', t='timestamp')
print(sample_gdf)
print(f'We create a TrajectoryCollection: {traj}')


print(traj.trajectories[0].df)
print('We show that only two rows remain due to the index being identical timestamps')

   row_id  id             geometry           timestamp
0       1   1  POINT (12.34 56.78) 2025-09-02 12:00:00
1       2   1  POINT (12.34 56.78) 2025-09-02 12:00:00
2       3   1  POINT (12.34 56.78) 2025-09-02 12:00:00
3       4   1  POINT (12.34 56.78) 2025-09-02 12:00:00
4       5   1  POINT (12.34 56.78) 2025-09-02 12:00:05
We create a TrajectoryCollection: TrajectoryCollection with 1 trajectories
                     row_id  id             geometry
timestamp                                           
2025-09-02 12:00:00       1   1  POINT (12.34 56.78)
2025-09-02 12:00:05       5   1  POINT (12.34 56.78)
We show that only two rows remain due to the index being identical timestamps


##### 3.4.2 Analyzing duplicates

We show that duplicate coordinates dont differ

In [7]:
# In this piece of code we create a column that codes for wether or not there is a duplicate
gdf["duplicate"] = gdf.duplicated(subset=["trajectory_id", "recordedAt"], keep=False)

dup_df = gdf[gdf["duplicate"]].copy()


# Step 3: iterate over each group of duplicates
num_coords = {}
len_coords = []
for _, group in dup_df.groupby(["trajectory_id", "recordedAt"]):
    traj_id = group['trajectory_id'].values[0]
    b = group['geometry'].values
    unique_coords = set(p.coords[0] for p in b)
    num_coords[traj_id] = unique_coords
    len_coords.append(len(unique_coords))

# Top three
sort = sorted(len_coords,reverse = True)
top_3 = sort[:3]

max_value_key = max(num_coords, key = lambda k : len(num_coords[k]))
max_values = num_coords[max_value_key]
print(f'top 3{top_3}')
print(max_values)
max_value_rows = gdf[(gdf['trajectory_id']==max_value_key) & ( gdf['duplicate'] == True)]
max_value_rows[max_value_rows['event']=='position']

# We see that only one pair has unidentical coordinates furthermore they look identical until printing every decimal.

top 3[2, 1, 1]
{(12.775742138831562, 56.00332263415917)}


,_id,recordedAt,deviceId,event,odometer,geometry,trajectory_id,duplicate
48133,678155f2ed570034f70cf724,2025-01-10 17:16:33.979,ALEWIL-11,position,NaN,POINT (12.82935 55.95652),000001,True
49806,678188f35ddf352bd6c7cc58,2025-01-10 20:54:11.766,ALEWIL-11,position,NaN,POINT (12.82935 55.95652),000001,True
57191,6783cb8d4ad51c5299d4379e,2025-01-12 14:02:52.838,ALEWIL-11,position,NaN,POINT (12.77623 56.00339),000001,True
57973,6783dffe5ddf352bd6cb5541,2025-01-12 15:30:06.360,ALEWIL-11,position,NaN,POINT (12.82935 55.95652),000001,True
104975,678e59038914f4fea64a2b49,2025-01-20 14:09:07.512,ALEWIL-11,position,NaN,POINT (12.96564 56.05316),000001,True
106016,678e654fd04529ef3b26348a,2025-01-20 15:01:34.998,ALEWIL-11,position,NaN,POINT (12.90319 56.07711),000001,True
106672,678e6d2f5ddf352bd6ee9524,2025-01-20 15:35:11.202,ALEWIL-11,position,NaN,POINT (12.77574 56.00332),000001,True


In [115]:
# This functions treats time duplicates within trajectories. We wish to keep one element in each group and prioritize as followes reg_started, reg_completed, reg_postponed, 'position', ... etc. 

def keep_prioritized_duplicates(df):
    drop_ids = []
    cnt = 0
    duplicate_df = df[df['duplicate']==True]
    for index, grp in duplicate_df.groupby(['trajectory_id','recordedAt']):
        cnt += 1
        print(f'GROUP NUMBER {cnt}')
        print(grp['event'].values)
        if 'reg_started' in grp['event'].values:
            print("HERE IS REG_STARTED")
            drops_event_with_id = grp[grp['event']!= 'reg_started'][['event','_id']].values.tolist()
            drops = grp[grp['event']!= 'reg_started'][['_id']].values.tolist()
            print(f'Events to drop {drops_event_with_id}')
            print(f'IDS to drop {drops}')
            drop_ids.append(drops)
            #print(f'ID APPENDED TO DROPLIST{drop_ids}')
        elif 'reg_complete' in grp['event'].values:
            print("HERE IS REG_COMPLETE")
            drops_event_with_id = grp[grp['event'] != 'reg_complete'][['event','_id']].values.tolist()
            drops = grp[grp['event'] != 'reg_complete'][['_id']].values.tolist()
            print(f'Events to drop {drops_event_with_id}')
            print(f'IDS to drop {drops}')
            drop_ids.append(drops)
            #print(f'ID APPENDED TO DROPLIST{drop_ids}')
        elif 'reg_postpone' in grp['event'].values:
            print("HERE IS REG_POSTPONE")
            drops_event_with_id = grp[grp['event'] != 'reg_postpone'][['event','_id']].values.tolist()
            drops = grp[grp['event'] != 'reg_postpone'][['_id']].values.tolist()
            print(f'Events to drop {drops_event_with_id}')
            print(f'IDS to drop {drops}')
            drop_ids.append(drops)
            #print(f'ID APPENDED TO DROPLIST{drop_ids}')
        elif 'position' in grp['event'].values:
            print("HERE IS A POSITION")
            drops_event_with_id = grp[grp['event'] != 'position'][['event','_id']].values.tolist()
            drops = grp[grp['event'] != 'position'][['_id']].values.tolist()
            print(f'Events to drop {drops_event_with_id}')
            print(f'IDS to drop {drops}')
            drop_ids.append(drops)
            #print(f'ID APPENDED TO DROPLIST{drop_ids}')
    d = [obj for sublist in drop_ids for item in sublist for obj in item]
    #print(f'unnpacking d:{d}')
    results = df[~df['_id'].isin(d)].reset_index(drop =True)
    return results


In [116]:
clean = keep_prioritized_duplicates(gdf)

GROUP NUMBER 1
['position' 'service_start' 'log_in']
HERE IS A POSITION
Events to drop [['service_start', ObjectId('678155f2d04529ef3bfd180e')], ['log_in', ObjectId('678155f35ddf352bd6c759bc')]]
IDS to drop [[ObjectId('678155f2d04529ef3bfd180e')], [ObjectId('678155f35ddf352bd6c759bc')]]
GROUP NUMBER 2
['position' 'service_stop']
HERE IS A POSITION
Events to drop [['service_stop', ObjectId('6783cb8d395bd082ec368e23')]]
IDS to drop [[ObjectId('6783cb8d395bd082ec368e23')]]
GROUP NUMBER 3
['position' 'service_start' 'log_in']
HERE IS A POSITION
Events to drop [['service_start', ObjectId('6783cb8e8914f4fea6250a2e')], ['log_in', ObjectId('6783cb8e5ddf352bd6cb1d91')]]
IDS to drop [[ObjectId('6783cb8e8914f4fea6250a2e')], [ObjectId('6783cb8e5ddf352bd6cb1d91')]]
GROUP NUMBER 4
['position' 'service_stop']
HERE IS A POSITION
Events to drop [['service_stop', ObjectId('678e5903a1b5f13bf7e89fbe')]]
IDS to drop [[ObjectId('678e5903a1b5f13bf7e89fbe')]]
GROUP NUMBER 5
['position' 'service_start' 'log_in

In [ ]:
# Now we look at duplicates that doenst have a prioritized event ("position", "reg_started", "reg_postpone", "reg_complete")
gdf["non_prioritized_event_duplicates"] = False  # Initialize a new column to store results
dup_df = gdf[gdf["duplicate"]].copy()
keep_values = ["position", "reg_started", "reg_postpone", "reg_complete"]

for _, group in dup_df.groupby(["trajectory_id", "recordedAt"]):
    # In duplicate groups with a prioritized event, we drop unprioritized events.
    gdf1 = gdf[~(
    (gdf['duplicate'] == True) &
    (gdf['non_position_event_duplicates'] == False) & (~gdf["event"].isin(keep_values).any()))]

# Now we decide h



In [ ]:
gdf.reset_index(drop=True, inplace=True)
gdf['stop_group'] = (~gdf['non_position_event_duplicates']).cumsum() * gdf['non_position_event_duplicates']
stop_sequences = gdf.loc[gdf['stop_group'] > 0, 'stop_group'].unique()
windows = []

for grp in stop_sequences:
    # Indices of this stop sequence
    idxs = gdf.index[gdf['stop_group'] == grp]
    #print(idxs)
    start_idx = max(idxs[0] - 10, 0)           # 10 rows before first stop
    #print(start_idx)
    end_idx = min(idxs[-1] + 5, len(gdf) - 1)   # 5 rows after last stop
    #print(end_idx)
    data = gdf.loc[start_idx:end_idx].copy()
    data['plot_id'] = data['stop_group'].max()
    data['color'] = data['stop_group'].apply(lambda x: 'black' if x == 0 else 'red')
    windows.append(data)

In [ ]:
# Preliminary documentation that all stop groups have identical coordinates at a given time.

In [ ]:
# Checking assignemtn of stop groups.
gdf['stop_group'].value_counts()

#We see that stop group 275847 has 17 rows. We check the corresponding trajectory and confirm results.
df = gdf[gdf['trajectory_id']=='003157']
#Since there is no "valid" events in a large part of the trajectory, the stop_group_id does not accumulate.

In [ ]:
# Creating HTML diagrams for 20 stop groups. Acces in the folder "non_position_duplicate_plots"
for i in range(0,20):
    plot = windows[i]
    line = LineString(plot.geometry.tolist())
    gdf_line = gpd.GeoDataFrame({"geometry": [line]}, crs="EPSG:4326")
    m = gdf_line.explore(weight = 1, color = 'black')
    g = plot.explore(m=m, color='color', marker_type = 'circle',
    categorical=True,
    m_size = 20,
    legend=False)
    #g.save(f"FIG_{i}")

In [132]:
vc = df1['event'].value_counts()
vc_df = vc.reset_index()
vc_df.columns = ['event', 'count']

In [135]:
print(vc_df.to_markdown(index=False))

| event            |   count |
|:-----------------|--------:|
| position         | 1118930 |
| pic_loc_warmup   |    6336 |
| service_stop     |    5857 |
| service_reset    |    5205 |
| reg_started      |    3799 |
| pos_sending_stop |    2499 |
| reg_complete     |    2304 |
| service_start    |    1588 |
| reg_postpone     |    1501 |
| log_in           |    1499 |
| log_out          |     252 |


In [129]:
vc_df.to_markdown(index=False)

'| event            |   count |\n|:-----------------|--------:|\n| position         | 1118930 |\n| pic_loc_warmup   |    6336 |\n| service_stop     |    5857 |\n| service_reset    |    5205 |\n| reg_started      |    3799 |\n| pos_sending_stop |    2499 |\n| reg_complete     |    2304 |\n| service_start    |    1588 |\n| reg_postpone     |    1501 |\n| log_in           |    1499 |\n| log_out          |     252 |'

## 4 Splitting trajectories on observationgap

### 4.1 Preparing sample data (two week interval)

In [ ]:
start_date = '2025-05-01'
end_date = '2025-05-14'

gdf_2weeks = gdf[(gdf['recordedAt'] >= start_date) & (gdf['recordedAt'] <= end_date)]


# Creating a trajectory collection
tc = mpd.TrajectoryCollection(gdf_2weeks, "trajectory_id", obj_id_col="deviceId", t="recordedAt")

print("TRAJECTORY COLLECTION DONE")


### 4.2 Splitting trajectories

In [ ]:
splits = [
    timedelta(days=4),
    timedelta(days=3),
    timedelta(days=2),
    timedelta(days=1),
    timedelta(hours=20),
    timedelta(hours=19),
    timedelta(hours=18),
    timedelta(hours=17),
    timedelta(hours=16),
    timedelta(hours=14),
    timedelta(hours=12),
    timedelta(hours=6),
    timedelta(hours=3),
    timedelta(hours=1),
    timedelta(minutes=30),
    timedelta(minutes=15),
    timedelta(minutes=5),
    timedelta(minutes=4),
    timedelta(minutes=3),
    timedelta(minutes=2),
    timedelta(minutes=1),
    timedelta(seconds=30),
    timedelta(seconds=15),
    timedelta(seconds=14),
    timedelta(seconds=13),
    timedelta(seconds=12),
    timedelta(seconds=11),
    timedelta(seconds=10),
    timedelta(seconds=9),
    timedelta(seconds=8),
    timedelta(seconds=7),
    timedelta(seconds=6),
    timedelta(seconds=5),
    timedelta(seconds=4),
    timedelta(seconds=3),
    timedelta(seconds=2),
    timedelta(seconds=1),
]


# MovingPandas Documentation of get_length

#Trajectory.get_length(units=(None, None, None, None))
#Return the length of the trajectory.

#Length is calculated using CRS units, except if the CRS is geographic (e.g. EPSG:4326 WGS84) then length is calculated in meters. 

#If units have been declared:

#For geographic projections, in declared units ( we have a geographic projection, so length is calculated in 'km' as declared below)


# See supported units at: https://movingpandas.org/units

def calculate_split_loss(trajcollection, split_list):
    total_length_no_split = round(sum(traj.get_length(units='km') for traj in trajcollection), 3)
    number_no_split = len(trajcollection)
    result_dict = {'no_split': (total_length_no_split, number_no_split)}

    for timedelta in split_list:
        split_tc = mpd.ObservationGapSplitter(trajcollection).split(gap=timedelta)
        total_length = round(sum(traj.get_length(units='km') for traj in split_tc), 3)
        number_of_trajectories = len(split_tc)
        result_dict[str(timedelta)] = (total_length, number_of_trajectories)
        print(f"Total length for gap {timedelta}: {total_length}, trajectories: {number_of_trajectories}")
    return result_dict

result_dict = calculate_split_loss(tc, splits)


### 4.3 Plotting distance loss

In [ ]:
# Separate no_split from the timed splits
no_split_length, no_split_count = result_dict.pop('no_split')
split_keys = list(result_dict.keys())
split_values = list(result_dict.values())
# Reinsert no_split into result_dict
result_dict = {'no_split': (no_split_length, no_split_count), **result_dict}

# Calculate percentage loss
split_percentages = [(1 - (val[0] / no_split_length)) * 100 for val in split_values]

# Number of trajectories
split_counts = [val[1] for val in split_values]

# Put no_split first
all_keys = ['no_split'] + split_keys
all_percentages = [0] + split_percentages  # no_split is 0% loss
all_counts = [no_split_count] + split_counts

# Create figure and first axis
fig, ax1 = plt.subplots(figsize=(12,6))

# Plot percentage loss on left y-axis
ax1.plot(all_keys, all_percentages, marker='o', color='black', label='Percentage loss')
ax1.set_xlabel('Timedelta for splitting trajectories')
ax1.set_xticks(range(len(all_keys)))
ax1.set_xticklabels(all_keys, rotation=90)
ax1.set_ylabel('Decrease in total trajectory length (%)', color='black')
ax1.set_yscale('log')
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True, ls='--', alpha=0.5)

# Create second axis for number of trajectories
ax2 = ax1.twinx()
ax2.plot(all_keys, all_counts, marker='s', color='blue', label='Number of trajectories')
ax2.set_ylabel('Number of trajectories', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

# Combine legends
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.xticks(rotation=45)
plt.title('Data loss and number of trajectories by split duration')
plt.savefig('splitloss.png')
plt.tight_layout()
plt.show()



In [ ]:

def plot_non_position_event_duplicates(traj_id, title = ""):
    traj_df = gdf[gdf['trajectory_id']== traj_id].copy()
    position_event_duplicates = traj_df[traj_df['position_event_duplicates']== True].copy()
    non_position_event_duplicates = traj_df[traj_df['position_event_duplicates']== False].copy()
    shortest_distances = non_position_event_duplicates.geometry.apply(
    lambda point: position_event_duplicates.geometry.distance(point).min()
    )
    shortest_time_differences = non_position_event_duplicates.recordedAt.apply(
    lambda time: (position_event_duplicates.recordedAt - time).abs().min())
    # Add distances as a new column
    non_position_event_duplicates["shortest_distance"] = shortest_distances
    non_position_event_duplicates["shortest_time_difference"] = shortest_time_differences
    # Convert timedelta to seconds
    non_position_event_duplicates['shortest_time_seconds'] = non_position_event_duplicates['shortest_time_difference'].dt.total_seconds()


    next_shortest_time_differences = non_position_event_duplicates.recordedAt.apply(
    lambda time: (position_event_duplicates.recordedAt - time).abs().nsmallest(2).iloc[1])
    non_position_event_duplicates["next_shortest_time_difference"] = next_shortest_time_differences.dt.total_seconds()

    fig, ax = plt.subplots(figsize=(10, 6))
    for x, y, time in zip(
        non_position_event_duplicates.geometry.x,
        non_position_event_duplicates.geometry.y,
        non_position_event_duplicates['next_shortest_time_difference']):
        ax.text(x, y, f"{time:.1f}", fontsize=8, ha='right', va='bottom', color = 'red')



    position_event_duplicates.plot(ax=ax, color = 'blue', label='Trajectory', marker='o', markersize=50)
    non_position_event_duplicates.plot(ax=ax, column = 'shortest_time_seconds', cmap = 'autumn_r', legend = True, label='Non-Position Event Duplicates', marker='x', markersize=50)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"{title} Trajectories with duplicates highlighted")
    ax.set_aspect('equal', adjustable='datalim')
    plt.legend()
    plt.show()

def print_traj_ids_with_non_position_event_duplicates(df):
    return df[df['position_event_duplicates'] == False]['trajectory_id'].value_counts()

top_10 = print_traj_ids_with_non_position_event_duplicates(gdf).head(10).index.tolist()
bottom_10 = print_traj_ids_with_non_position_event_duplicates(gdf).tail(10).index.tolist()
for traj_id in bottom_10:
    plot_non_position_event_duplicates(traj_id,title = f'Trajectory: {traj_id}, Bottom 10 -')
for traj_id in top_10:
    plot_non_position_event_duplicates(traj_id,title = f'Trajectory: {traj_id}, Top 10 -')
traj_df = gdf[gdf['trajectory_id']== '003183']
position_event_duplicates = traj_df[traj_df['position_event_duplicates']== True]
non_position_event_duplicates = traj_df[traj_df['position_event_duplicates']== False]
shortest_distances = non_position_event_duplicates.geometry.apply(
lambda point: position_event_duplicates.geometry.distance(point).min()
)

# Add distances as a new column
non_position_event_duplicates["shortest_distance"] = shortest_distances

print(non_position_event_duplicates[["geometry", "shortest_distance"]])
position_event_duplicates = traj_df[traj_df['position_event_duplicates']== True]
non_position_event_duplicates = traj_df[traj_df['position_event_duplicates']== False]
fig, ax = plt.subplots(figsize=(10, 6))
position_event_duplicates.plot(ax=ax, color='blue', label='Position Event Duplicates', marker='o', markersize=50)
non_position_event_duplicates.plot(ax=ax, color='red', label='Non-Position Event Duplicates', marker='x', markersize=50)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Trajectories with duplicates highlighted")
ax.set_aspect('equal', adjustable='datalim')
plt.legend()
plt.show()
keep_values = ["position", "reg_started", "reg_postpone", "reg_complete"]
for _, group in gdf_duplicates.groupby(["trajectory_id", "recordedAt"]):
    print(_)
    print(group[['recordedAt','trajectory_id','event']])
    if group['event'].isin(keep_values).any():
        print("  --> ASSIGNING TRUE")
        group['drop'] = True
    else:
        group['drop'] = False
        print("  --> ASSIGNING FALSE")

result = duplicates.groupby(["trajectory_id", "recordedAt"])["event"].apply(
    lambda x: x.isin(keep_values).any()
)
keep_values = ["position", "reg_started", "reg_postpone","reg_complete"]

result = duplicates.groupby(["trajectory_id", "recordedAt"])["event"].apply(
    lambda x: x.isin(keep_values).any()
)

print(f"Number of duplicate groups with values we want to keep {len(result[result == True])}")
print(f"Number of duplicate groups without values we want to keep {len(result[result == False])}")
print(f"Total {len(result)}")